# From IBGE survey files to BigQuery
FIAP postgraduate program in Data Analytics · Tech Challenge 3

Run this notebook in a Dataproc Spark Connect environment with the Spark–BigQuery connector. Upload the IBGE monthly CSV files and the English survey dictionary as described in the project README. Each run replaces the bronze and silver outputs and the two destination tables in the selected project.

In [ ]:
import os
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from pyspark.sql import functions as F

bucket = os.environ["PNAD_GCS_BUCKET"]
dataset = os.environ["PNAD_BQ_DATASET"]
lake = f"gs://{bucket}/pnad"
spark = DataprocSparkSession.builder.getOrCreate()

## 1. Land the inputs in the bronze layer
Read source codes as strings so month and category codes keep their leading zeros.

In [ ]:
raw_survey = spark.read.option("header", True).csv(f"{lake}/raw/survey/")
raw_dictionary = spark.read.option("header", True).csv(f"{lake}/raw/dictionary/")
for name, records in [("survey", raw_survey), ("dictionary", raw_dictionary)]:
    records.write.mode("overwrite").parquet(f"{lake}/bronze/{name}/")

## 2. Prepare the English dictionary and analytical fields
The published dictionary has one row per selected field and response code, with its question and response already spelled out in English. The raw microdata retain the IBGE field identifiers, including the reference-year field. Continuous fields retain their values. Small lookup tables supply the English response labels for categorical fields.

In [ ]:
dictionary = spark.read.parquet(f"{lake}/bronze/dictionary/").filter(F.col("selection") == "OK")
raw = spark.read.parquet(f"{lake}/bronze/survey/")
source_fields = {'Ano': 'survey_year',
 'UF': 'state',
 'CAPITAL': 'state_capital',
 'V1013': 'survey_month',
 'V1022': 'urban_rural',
 'A002': 'age',
 'A003': 'sex',
 'A005': 'education',
 'B0011': 'fever',
 'B0012': 'cough',
 'B0014': 'difficulty_breathing',
 'B00111': 'loss_of_smell_or_taste',
 'B002': 'sought_healthcare',
 'B0031': 'stayed_home_when_ill',
 'B008': 'took_covid_test',
 'B009A': 'took_swab_test',
 'B009B': 'swab_result',
 'B009C': 'took_finger_prick_test',
 'B009D': 'finger_prick_result',
 'B009E': 'took_venous_blood_test',
 'B009F': 'venous_blood_result',
 'B011': 'social_distancing',
 'C01012': 'cash_earnings',
 'E001': 'household_loan'}
continuous = {"Ano", "V1013", "A002", "C01012"}
survey = raw.select(*source_fields)
for source, field in source_fields.items():
    if source in continuous:
        survey = survey.withColumnRenamed(source, field)
        continue
    lookup = dictionary.filter(F.col("source_variable") == source).select(
        F.col("response_code").alias("lookup_code"), F.col("response_label").alias("lookup_label"))
    survey = (survey.join(F.broadcast(lookup), survey[source] == F.col("lookup_code"), "left")
              .drop(source, "lookup_code").withColumnRenamed("lookup_label", field))
survey = (survey.withColumn("age", F.col("age").cast("long"))
          .withColumn("cash_earnings", F.col("cash_earnings").cast("long"))
          .withColumn("survey_month", F.lpad("survey_month", 2, "0")))
dictionary.write.mode("overwrite").parquet(f"{lake}/silver/dictionary/")
survey.write.mode("overwrite").parquet(f"{lake}/silver/survey/")

## 3. Publish the gold tables
The local analysis can read the resulting survey table through the optional BigQuery setting. The packaged Power BI snapshot also has a row identifier and the survey reference date, which were added for the dashboard.

In [ ]:
for table_name, folder in [("survey", "survey"), ("survey_dictionary", "dictionary")]:
    records = spark.read.parquet(f"{lake}/silver/{folder}/")
    (records.write.format("bigquery")
     .option("table", f"{dataset}.{table_name}")
     .option("temporaryGcsBucket", bucket)
     .mode("overwrite").save())